# Benchmark: the numbers

Scores checkpoints two ways. The config picks the tiles, so the same notebook serves the val strip (tuning and diagnosis) and the test strip (final score, once).

1. **Crops, with clicks from the true labels.** One tree at a time in a 12 m crop, no merging. This measures mask quality given a good click.
2. **Whole tiles, as in real use.** Ground removal, clicks, one mask per click, merging, then F1 at IoU 0.5.

**Tile scores are reported at three points, because low F1 can come from the clicks, the masks or the merging:**

- `click_recall`: share of trees with a click on them.
- `raw_recall`: share of trees where some single mask reaches IoU 0.5, **before merging**.
- `recall`, `precision`, `f1`: after merging, with one prediction matched to one tree.

**Click sources:** `perfect` is one click per tree from the true labels, the best any click finder could do. `treex` and `stems` are automatic. A checkpoint named `treex_full` is the baseline: TreeX segmenting the tile by itself, with no model.

**Two scoring rules:** `strict` counts every unmatched prediction as wrong. `ignore_non_tree` skips predictions mostly on non-tree points, since some of those are real understory without a label.

`mean_tree_iou` is each real tree's best IoU with any prediction, missed trees counting as 0. `mean_matched_iou` is for matched trees only.

**Score the test strip once, after choosing the checkpoint.** Tune on the val strip (`diagnose_val.yaml`). Needs a GPU. To see one tile go through every stage, use `04_inference_steps.ipynb`.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from torch.utils.data import DataLoader

from treeseg.data.dataset import ForestCrops
from treeseg.engine.benchmark import GROUND_CACHE, prepare_tile, run_benchmark, summarise
from treeseg.engine.click_demo import predict_one_click
from treeseg.engine.scoring import match_instances
from treeseg.engine.trainer import collate, validate
from treeseg.models.point_sam import add_point_sam_to_path, build_point_sam
from treeseg.utils.curves import plot_f1_bars
from treeseg.utils.view3d import plot_mask_candidates, plot_outcome, plot_tree_gallery

cfg = yaml.safe_load(open(os.environ.get("TREESEG_CONFIG", "configs/experiment/benchmark.yaml")))
os.makedirs(cfg["out_dir"], exist_ok=True)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: this notebook will fail")
cfg

## 1. Crop scores: how good are the masks, given a good click?

Runs each checkpoint on the crops of the strip named by `crop_split` in the config (skipped if the config has none). Each first round is 2 trunk clicks placed using the true labels.

- `val_iou_pick`: IoU of the mask the model itself chooses from that first round, which is how real use chooses.
- `val_iou_first`: IoU of the best of those 3 masks, given the true answer.
- `val_iou`: IoU after correction clicks.

(The names say `val` because they come from the same function as validation.)

In [ ]:
crop_split = cfg.get("crop_split")
if crop_split:
    add_point_sam_to_path()
    from pc_sam.model.loss import Criterion

    crop_set = ForestCrops(f"data/wytham_isprs/crops/{crop_split}", 10000, 4, 40, seed=2)
    loader = DataLoader(crop_set, batch_size=1, shuffle=False, collate_fn=collate)
    print(f"{crop_split} crops:", len(crop_set))
    EXAMPLE_CROPS = (5, 20, 40)
    rows, examples = [], {}
    for name, path in cfg["checkpoints"].items():
        model = build_point_sam(path, cfg["group_number"], cfg["group_size"]).to("cuda")
        rows.append({"checkpoint": name, **validate(model, loader, Criterion(use_soft_iou=False), "cuda", "stem")})
        #the same few trees for each model, so the pictures can be compared
        examples[name] = [predict_one_click(model, crop_set[index], 0, "stem1") for index in EXAMPLE_CROPS]
        del model
        torch.cuda.empty_cache()
    crop_scores = pd.DataFrame(rows).round(3)
    crop_scores.to_csv(os.path.join(cfg["out_dir"], f"crop_scores_{crop_split}.csv"), index=False)
    display(crop_scores)
else:
    print("no crop_split in this config, so crop scores are skipped")

The true tree, then the 3 masks for one trunk click, the green diamond. Compare models for the same crop.

In [ ]:
if crop_split:
    for number, index in enumerate(EXAMPLE_CROPS):
        for name in examples:
            plot_mask_candidates(examples[name][number], f"{crop_split} crop {index}: {name}")
            plt.savefig(os.path.join(cfg["out_dir"], f"crop{index}_{name}.png"))
            plt.show()

## 2. Tile scores

Rows are saved to `results.csv` as they finish, and each prediction is saved as `.npy`. With `save_masks: true` in the config, every raw mask is saved too, so `04_inference_steps.ipynb` can replay merging without a GPU.

In [ ]:
results_path = run_benchmark(cfg)

### Results

In [ ]:
results_path = os.path.join(cfg["out_dir"], "results.csv")
summary = summarise(results_path)
summary.to_csv(os.path.join(cfg["out_dir"], "summary.csv"), index=False)
summary

In [ ]:
figure_path = os.path.join(cfg["out_dir"], "f1_bars.png")
plot_f1_bars(summary, figure_path)
display(Image(figure_path))

### Clicks, masks or merging?

Read each row left to right: trees with a click, trees with a good mask before merging, trees matched after merging.

- `raw_recall` low even with `perfect` clicks: the masks are the limit.
- `raw_recall` high but `recall` much lower: merging is losing trees.
- `perfect` high and `treex` low: the clicks are the limit.

The `treex_full` row has no masks to merge, so it only has the last number.

In [ ]:
stages = summary[summary["rule"] == "ignore_non_tree"][["checkpoint", "clicks_from", "click_recall", "raw_recall", "recall", "precision", "f1"]]
stages.to_csv(os.path.join(cfg["out_dir"], "stages.csv"), index=False)
stages

### Per tile

Averages can hide one tile doing all the work.

In [ ]:
per_tile = pd.read_csv(results_path)
per_tile[per_tile["rule"] == "ignore_non_tree"][["checkpoint", "tile", "clicks_from", "f1", "mean_tree_iou", "mean_matched_iou", "precision", "raw_recall", "recall", "trees", "predictions", "click_recall"]].round(3)

## See it

True trees next to predicted trees for one result. A matched tree has the same colour on both sides, true trees with no match are dark, and predictions matching nothing are red. Change the three settings to look at other results. `04_inference_steps.ipynb` shows how a tile gets here, stage by stage.

In [ ]:
#the last model row in the results: a checkpoint, plus a variant name if the config has variants
CHECKPOINT = [name for name in pd.read_csv(results_path)["checkpoint"].unique() if name not in cfg.get("baselines", [])][-1]
TILE, CLICKS = cfg["tiles"][0], cfg["click_methods"][0]

#the saved ground removal, so the points here are the ones the prediction was made on
tile = prepare_tile(cfg["tile_dir"], TILE, cfg["csf"], cfg["min_tree_points"], cache_dir=cfg.get("ground_cache", GROUND_CACHE))
pred = np.load(os.path.join(cfg["out_dir"], f"{CHECKPOINT}_{TILE}_{CLICKS}_pred.npy"))
match = match_instances(tile["ids"], pred, tile["sizes"], cfg["iou_threshold"], ignore_non_tree_preds=True)
plot_outcome(tile["xyz"], tile["ids"], pred, match, title=f"{CHECKPOINT}, {TILE}, {CLICKS} clicks")
plt.savefig(os.path.join(cfg["out_dir"], f"outcome_{CHECKPOINT}_{TILE}_{CLICKS}.png"))
plt.show()

In [ ]:
plot_tree_gallery(tile["xyz"], tile["ids"], pred, match, f"{CHECKPOINT}, {TILE}, {CLICKS} clicks")
plt.savefig(os.path.join(cfg["out_dir"], f"gallery_{CHECKPOINT}_{TILE}_{CLICKS}.png"))
plt.show()